Auto Loader itself cannot directly read Excel files sheet-wise like CSV, JSON, or Parquet because Auto Loader primarily supports:

CSV
JSON
Parquet
Avro
XML (with library support)

For Excel (.xlsx), the common approach is:

Step 1: Use Auto Loader to detect Excel files

In [0]:
stream_df = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "binaryFile")
         .load("/Volumes/databricks_learning/bdx/raw_data")
)
# This ingests the Excel files as binary data.

Step 2: Read specific sheet using spark-excel

Install:

com.crealytics:spark-excel

In [0]:
# read specific sheet
stream_df = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "binaryFile")
         .option("cloudFiles.schemaLocation", "dbfs:/tmp/flight/schema")
         .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
         .option("cloudFiles.schemaHints", "content string")
         .load("/Volumes/databricks_learning/bdx/raw_data")
)
df_files = (
    spark.read
         .format("binaryFile")
         .load("/Volumes/databricks_learning/bdx/raw_data")
)

In [0]:
df_files_batch = (
    spark.read
         .format("binaryFile")
         .load("/Volumes/databricks_learning/bdx/raw_data")
)
file_paths = [row.path for row in df_files_batch.select("path").collect()]
file_paths

# 📘 Loading Excel (.xlsx) Sheet Data into a Bronze Table using Auto Loader

## Overview

Auto Loader (`cloudFiles`) does **not** natively support Excel files — only CSV, JSON, Parquet, Avro, and XML. The proven pattern on Databricks is a **two-stage pipeline**:

1. **Ingest** — Auto Loader streams the `.xlsx` files as **binary files** into a raw/binary bronze layer (or keeps them as paths).
2. **Parse** — inside a `foreachBatch` function, each new file is read **sheet-wise** with the **spark-excel** library (or pandas + openpyxl) and appended to the structured bronze Delta table.

```
.xlsx files in Volume
        │
        ▼  (Auto Loader, binaryFile format)
  binary bronze table  ── foreachBatch ──►  spark-excel (sheet-level read)
                                              │
                                              ▼
                                    bronze Delta table (parsed rows)
```

## Prerequisites

Install the spark-excel library on the cluster (or as a notebook-scoped Maven library):

| Field | Value |
|---|---|
| Maven coordinates | `com.crealytics:spark-excel_2.12:<version>` (match your Spark version) |
| Databricks runtime | Works on DBR with Spark 2.12/3.x scale artifacts |

## Step 1 — Stream Excel files in as binary with Auto Loader

Auto Loader detects new `.xlsx` files and hands each one over as a row containing its raw `content` bytes, `path`, and `modificationTime`. The schema location and checkpoint location guarantee **exactly-once** processing — a file already ingested is never re-processed.

```python
source_path   = "/Volumes/databricks_learning/bdx/raw_data"
checkpoint    = "dbfs:/tmp/flight/checkpoint"
schema_loc    = "dbfs:/tmp/flight/schema"
bronze_table  = "flight_bronde"  # e.g. catalog.schema.flight_bronze

binary_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "binaryFile")           # ingest raw bytes
         .option("cloudFiles.includeExistingFiles", "true")   # backfill existing files
         .option("cloudFiles.schemaLocation", schema_loc)      # tracks discovered files
         .load(source_path)
)
```

## Step 2 — Parse each file's specific sheet inside `foreachBatch`

Auto Loader only streams; it cannot parse Excel. So we use `foreachBatch`, which is triggered for **every micro-batch of new files**. Inside it we loop over the batch's file paths and read the **named sheet** with spark-excel in **batch** mode (spark-excel has no streaming reader — that is exactly why this two-stage pattern exists).

```python
from pyspark.sql import functions as F

sheet_name = "Sheet1"  # or "FlightData", or a data address like "'FlightData'!A1:H100"

def parse_excel_batch(df, batch_id):
    """Called once per micro-batch with the new binary files."""
    paths = [row["path"] for row in df.select("path").distinct().collect()]

    parsed_dfs = []
    for p in paths:
        excel_df = (
            spark.read
                 .format("com.crealytics.spark.excel")
                 .option("header", "true")                # first row = column names
                 .option("dataAddress", f"'{sheet_name}'!A1")  # <-- specific sheet
                 .option("inferSchema", "true")
                 .load(p)
        )
        # Attach provenance / audit columns
        excel_df = (
            excel_df
            .withColumn("_source_file", F.lit(p))
            .withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_batch_id", F.lit(batch_id))
        )
        parsed_dfs.append(excel_df)

    if parsed_dfs:
        from functools import reduce
        reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), parsed_dfs) \
            .write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(bronze_table)

query = (
    binary_stream
        .writeStream
        .foreachBatch(parse_excel_batch)
        .option("checkpointLocation", checkpoint)  # exactly-once guarantee
        .trigger(availableNow=True)                 # run once, then stop (batch-style)
        .start()
)
query.awaitTermination()
```

> **Tip — Trigger mode:** use `trigger(availableNow=True)` for a job-like run (process everything available and stop), or the default micro-batch trigger for a continuous stream.

## Step 3 — Alternative: pandas + openpyxl (driver-side, no Maven library)

If you cannot install spark-excel, parse the binary `content` with pandas inside `foreachBatch`:

```python
import io
import pandas as pd

def parse_excel_with_pandas(df, batch_id):
    rows = df.select("path", "content").collect()
    pdf_list = []
    for r in rows:
        pdf = pd.read_excel(io.BytesIO(r["content"]), sheet_name=sheet_name)
        pdf["_source_file"] = r["path"]
        pdf_list.append(pdf)
    if pdf_list:
        spark.createDataFrame(pd.concat(pdf_list, ignore_index=True)) \
             .withColumn("_ingested_at", F.current_timestamp()) \
             .write.format("delta").mode("append").saveAsTable(bronze_table)
```
This is fine for small/medium files; for large workbooks prefer the distributed spark-excel approach.

## How the exactly-once guarantee works

| Component | Role |
|---|---|
| `checkpointLocation` | Stores which files/offsets were processed — restarts skip them |
| `cloudFiles.schemaLocation` | Tracks discovered files & the binary schema across restarts |
| Delta `append` mode | Idempotent per batch when combined with the checkpoint |

## Bronze table conventions

- Keep raw values as-is (do **not** cast or clean — that's silver's job).
- Add audit columns: `_source_file`, `_ingested_at`, `_batch_id`.
- Enable Delta features like `delta.appendOnly` or CDF if downstream silver layers need change tracking.

## Common pitfalls

1. **Sheet name mismatch** — `dataAddress` must match the sheet name exactly (case-sensitive, quotes around names with spaces).
2. **Driver OOM with pandas path** — `collect()` of binary content loads all files onto the driver; keep batches small or use spark-excel.
3. **Schema drift across files** — Excel has no strict schema; use `unionByName(..., allowMissingColumns=True)` and cast columns in silver.
4. **Files arriving mid-write** — Auto Loader only picks up *complete* files; if a producer writes files slowly, write them with a temp suffix and rename, or use file notification modes.
5. **Duplicate sheet data** — if a file is re-uploaded under a new name, it is re-parsed; dedupe in silver with a Delta `MERGE`.

In [0]:
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "excel")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/databricks_learning/bdx/raw_data")
)
df.show()

In [0]:
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "excel")
    .option("cloudFiles.schemaEvolutionMode", "none")
    .option("header", "true")
    .load("/Volumes/databricks_learning/bdx/raw_data")
)
df.show()

In [0]:
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "excel")
    .option(
        "cloudFiles.schemaLocation",
        "/Volumes/databricks_learning/bdx/raw_data/_excel_schema"
    )
    .option("cloudFiles.schemaEvolutionMode", "none")
    .option("header", "true")
    .option("startAt", "B6")
    .load("/Volumes/databricks_learning/bdx/raw_data")
)
query = (
    df.writeStream
    .format("memory")
    .queryName("excel_preview")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/databricks_learning/bdx/raw_data/_excel_checkpoint_v4")
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()

display(spark.table("excel_preview"))

In [0]:
# PRODUCTION-READY: Auto Loader ingest of the "Healthcare Insurance" sheet from
# .xlsx files -> Bronze Delta table. Built for scheduled execution via Databricks
# Jobs: every knob is a notebook widget, so a Job task can override any of them
# via its parameters, e.g. {"source_path": "...", "trigger_mode": "availableNow"}.
#
# Key design decisions (hard-won from earlier debugging):
# - Auto Loader's excel schema INFERENCE never picked up the header row (it
#   produced _c0/_c1 placeholders), so an EXPLICIT schema is built from the real
#   header row (B6) of one existing file. Inference is bypassed completely.
# - `cloudFiles.rescuedDataColumn` is CSV/JSON only; for excel it is silently
#   ignored — not set, schema drift is checked downstream in silver instead.
# - "startAt" is not a valid option (silently ignored); dataAddress with an
#   exact sheet!range is used instead.
# - The memory sink was replaced with a durable Delta sink (foreachBatch) plus
#   bronze audit columns (_source_file, _ingested_at, _batch_id). Checkpoint +
#   schema location give exactly-once processing across Job runs.

import logging
import re

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

logger = logging.getLogger("excel_bronze_ingest")
logger.info("Starting Excel bronze ingestion job")

# ---------------------------------------------------------------------------
# Parameters — overridable from Jobs (Job task parameters map to widget names)
# ---------------------------------------------------------------------------
dbutils.widgets.text("source_path", "/Volumes/databricks_learning/bdx/raw_data")
dbutils.widgets.text("target_table", "databricks_learning.bdx.bronze_healthcare_insurance")
dbutils.widgets.text("sheet_name", "Healthcare Insurance")
dbutils.widgets.text("header_cell", "B6")            # top-left cell of the header row
dbutils.widgets.text("last_column", "I")             # last column of the data block
dbutils.widgets.text("max_rows", "10000")             # safety upper bound on sheet rows
dbutils.widgets.text("schema_location", "/Volumes/databricks_learning/bdx/raw_data/_excel_schema_prod_v2")
dbutils.widgets.text("checkpoint_location", "/Volumes/databricks_learning/bdx/raw_data/_excel_checkpoint_prod_v2")
dbutils.widgets.text("trigger_mode", "availableNow")  # availableNow (batch) | continuous

SOURCE_PATH         = dbutils.widgets.get("source_path").rstrip("/")
TARGET_TABLE        = dbutils.widgets.get("target_table")
SHEET_NAME          = dbutils.widgets.get("sheet_name")
HEADER_CELL         = dbutils.widgets.get("header_cell")
LAST_COLUMN         = dbutils.widgets.get("last_column")
MAX_ROWS            = int(dbutils.widgets.get("max_rows"))
SCHEMA_LOCATION     = dbutils.widgets.get("schema_location")
CHECKPOINT_LOCATION = dbutils.widgets.get("checkpoint_location")
TRIGGER_MODE        = dbutils.widgets.get("trigger_mode")

# ---------------------------------------------------------------------------
# Derive header and data ranges from the header cell
# (e.g. header_cell=B6, last_column=I -> header "...!B6:I6", data "...!B7:I10006")
# ---------------------------------------------------------------------------
m = re.fullmatch(r"([A-Z]+)(\d+)", HEADER_CELL.strip().upper())
if not m:
    raise ValueError(f"Invalid header_cell parameter {HEADER_CELL!r} — expected like 'B6'")
header_col, header_row = m.group(1), int(m.group(2))
HEADER_RANGE = f"{SHEET_NAME}!{header_col}{header_row}:{LAST_COLUMN}{header_row}"
DATA_RANGE   = f"{SHEET_NAME}!{header_col}{header_row + 1}:{LAST_COLUMN}{header_row + MAX_ROWS}"
logger.info("Header range: %s | Data range: %s", HEADER_RANGE, DATA_RANGE)

# ---------------------------------------------------------------------------
# Step 1 — discover one existing .xlsx file to read the header row from
# ---------------------------------------------------------------------------
sample = (
    spark.read.format("binaryFile")
    .load(SOURCE_PATH)
    .where("path LIKE '%.xlsx'")
    .select("path")
    .limit(1)
    .collect()
)
if not sample:
    raise ValueError(f"No .xlsx files found under {SOURCE_PATH}")
sample_path = sample[0]["path"]

# ---------------------------------------------------------------------------
# Step 2 — read ONLY the header row (as data, header=false) and build the
# EXPLICIT schema. This bypasses Auto Loader's broken excel inference — the
# root cause of the _c0/_c1 placeholder columns — permanently.
# ---------------------------------------------------------------------------
header_row_df = (
    spark.read.format("excel")
    .option("header", "false")
    .option("inferSchema", "false")
    .option("dataAddress", HEADER_RANGE)
    .load(sample_path)
    .collect()
)
if not header_row_df:
    raise ValueError(f"No header row found at {HEADER_RANGE} in {sample_path}")

header_names = [
    re.sub(r'[ ,;{}()\n\t=]+', '_', str(v).strip()) if v is not None and str(v).strip() else f"column_{i}"
    for i, v in enumerate(header_row_df[0])
]
logger.info("Detected %d header columns in %s: %s", len(header_names), sample_path, header_names)

# Bronze keeps all values as raw strings; casting/cleaning is silver's job.
explicit_schema = StructType([StructField(name, StringType(), True) for name in header_names])

# ---------------------------------------------------------------------------
# Step 3 — streaming read with Auto Loader
# ---------------------------------------------------------------------------
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "excel")                 # parse .xlsx directly (Apache POI)
    .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)  # tracks discovered files -> exactly-once discovery
    .option("cloudFiles.schemaEvolutionMode", "none")       # schema frozen to the explicit schema below
    .option("cloudFiles.includeExistingFiles", "true")      # backfill existing files on the first run
    .option("header", "true")                               # first row of the data range is the header
    .option("dataAddress", DATA_RANGE)                      # exact sheet + cell range (not "startAt")
    .schema(explicit_schema)
    .load(SOURCE_PATH)
)

# The data range end row (header_row + MAX_ROWS) is only a safety bound —
# drop the all-null padding rows beyond the actual data using the first REAL column.
first_col = header_names[0]
df = df.filter(F.col(first_col).isNotNull())

# ---------------------------------------------------------------------------
# Step 4 — durable Delta sink (foreachBatch) with provenance/audit columns
# ---------------------------------------------------------------------------
def write_bronze_batch(batch_df, batch_id):
    """Append one micro-batch to the bronze Delta table with audit columns."""
    out = (
        batch_df
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_batch_id", F.lit(batch_id))
    )
    # _metadata.file_path (Auto Loader file metadata) gives per-row provenance;
    # guard with a fallback in case the excel reader does not expose it.
    if "_metadata" in out.columns:
        out = out.withColumn("_source_file", F.col("_metadata.file_path"))
    (
        out.write
        .format("delta")
        .mode("append")
        .saveAsTable(TARGET_TABLE)
    )
    logger.info("Batch %d appended to %s", batch_id, TARGET_TABLE)

writer = (
    df.writeStream
    .format("delta")
    .foreachBatch(write_bronze_batch)
    .option("checkpointLocation", CHECKPOINT_LOCATION)  # committed files are never reprocessed
    .queryName("excel_bronze_ingest")
)

# ---------------------------------------------------------------------------
# Step 5 — trigger selection and run
# ---------------------------------------------------------------------------
if TRIGGER_MODE == "availableNow":
    # Process every currently available file, then terminate — ideal for
    # scheduled Jobs (behaves like a batch job with streaming guarantees).
    query = writer.trigger(availableNow=True).start()
elif TRIGGER_MODE == "continuous":
    # Long-running stream that picks up new files every minute — use with a
    # Jobs continuous task or a dedicated all-purpose cluster.
    query = writer.trigger(processingTime="1 minute").start()
else:
    raise ValueError(f"Invalid trigger_mode {TRIGGER_MODE!r} — use 'availableNow' or 'continuous'")

query.awaitTermination()

# ---------------------------------------------------------------------------
# Post-run verification / summary (visible in Job logs)
# ---------------------------------------------------------------------------
result = spark.table(TARGET_TABLE)
row_count = result.count()
logger.info("Run complete. Bronze table %s now holds %d rows.", TARGET_TABLE, row_count)
print(f"✅ Ingestion finished — {TARGET_TABLE} contains {row_count} rows.")
print("Columns:", result.columns)
display(result.limit(50))
